In [1]:
import torch
torch.manual_seed(0)

In [2]:
d = 4 # dimension of embedding = 128
w = 6 # width 
k = 2 # how many values we will consider for sparsification

n_tokens = 3

In [4]:
# sampling anchor projection from guassian dist as we need rotationally symmetric values
B = torch.randn(d, w) # with emb dim and width 
print("B", B.shape)

B torch.Size([4, 6])


In [6]:
# normalizing each col to unit length
col_norms = B.norm(dim=0, keepdim=True)
print("col_norms", col_norms.shape)

col_norms torch.Size([1, 6])


In [7]:
B = B / col_norms
print("unit columns?", B.norm(dim=0))

unit columns? tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [8]:
# project 
x = torch.randn(n_tokens, d)
x = x / x.norm(dim=1, keepdim=True)

p = x @ B
print("p", p.shape)
print(p)

p torch.Size([3, 6])
tensor([[ 0.4155,  0.2295,  0.5835, -0.5309,  0.2457, -0.1982],
        [-0.0898,  0.3983, -0.1933, -0.4063, -0.1674,  0.0563],
        [-0.0504, -0.1650,  0.6323, -0.1002, -0.0898, -0.5435]])


In [9]:
# above each row represents the projection of an emb at row_i to a B projected sphere

In [10]:
# Sparsification
values, indices = torch.topk(p, k, dim=-1)
print("values", values.shape, "\n", values)
print("indices", indices.shape, "\n", indices)

values torch.Size([3, 2]) 
 tensor([[ 0.5835,  0.4155],
        [ 0.3983,  0.0563],
        [ 0.6323, -0.0504]])
indices torch.Size([3, 2]) 
 tensor([[2, 0],
        [1, 5],
        [2, 0]])


In [13]:
sparse = torch.zeros_like(p)
print(f"Sparse before: {sparse}")
sparse.scatter_(dim=-1, index=indices, src=values)
print("sparse\n", sparse)

Sparse before: tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
sparse
 tensor([[ 0.4155,  0.0000,  0.5835,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.3983,  0.0000,  0.0000,  0.0000,  0.0563],
        [-0.0504,  0.0000,  0.6323,  0.0000,  0.0000,  0.0000]])


In [15]:
# pool
pooled = sparse.sum(dim=0)
print("pooled", pooled.shape)
print("Pooled values", pooled)

pooled torch.Size([6])
Pooled values tensor([0.3651, 0.3983, 1.2159, 0.0000, 0.0000, 0.0563])


In [18]:
mask = torch.zeros_like(p)
print(mask.shape)
print(mask)

torch.Size([3, 6])
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])


In [19]:
mask.scatter_(dim=-1, index=indices, src=torch.ones_like(values))
print(mask)

tensor([[1., 0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0., 1.],
        [1., 0., 1., 0., 0., 0.]])


In [20]:
counts = mask.sum(dim=0).clamp(min=1)
counts

tensor([2., 1., 2., 1., 1., 1.])

In [22]:
doc_vec = pooled / counts
print("pooled", pooled)
print("------"*5)
print("counts", counts)
print("======"*5)
print("docs", doc_vec)
print("doc_vec", doc_vec.shape)


pooled tensor([0.3651, 0.3983, 1.2159, 0.0000, 0.0000, 0.0563])
------------------------------
counts tensor([2., 1., 2., 1., 1., 1.])
docs tensor([0.1826, 0.3983, 0.6079, 0.0000, 0.0000, 0.0563])
doc_vec torch.Size([6])


# Naive implementation of SMVE

In [29]:
import numpy as np
import torch
from pathlib import Path
from smve_lab.config import ARTIFACTS_DIR
from smve_lab.storage import load_separate, get_sparse, get_colbert

d = 1024
w = 65536
k = 32 

B = torch.randn(d, w)
col_norms = B.norm(dim=0, keepdim=True)
B = B / col_norms # column normed

def smve(token_embeddings: torch.Tensor , k: int, B: torch.Tensor, is_query: bool) -> torch.Tensor :

    if not isinstance(token_embeddings, torch.Tensor):
        # np.asarray on a memmap is a cheap view; copy() makes it writable/contiguous
        token_embeddings = torch.from_numpy(np.array(token_embeddings, dtype=np.float16))
    token_embeddings = token_embeddings.to(B.dtype)

    # step 1: Random Projection onto Spherical Anchors
    # token_embeddinng : n_tokens x d
    # B -> Anchor projection : d x w

    projections = token_embeddings @ B # n_tokens x w

    # step 2: Sparsification
    values, indices = torch.topk(projections, k, dim=-1) # n_token x k -> both values and indices

    sparse = torch.zeros_like(projections) # n_tokens x w
    sparse.scatter_(dim=-1, index=indices, src=values) 

    # step 3: Pooling
    pooled = sparse.sum(dim=0)

    if is_query:
        return pooled

    mask = torch.zeros_like(projections) # n_tokens x w
    mask.scatter_(dim=-1, index=indices, src=torch.ones_like(values))
    counts = mask.sum(dim=0).clamp(min=1)

    doc_vec = pooled / counts

    return doc_vec


# test smve function
save_dir = ARTIFACTS_DIR / "embeddings" / "scifact_bgem3"
doc_emb, doc_ids = load_separate(save_dir, "docs", mmap_colbert=True)
query_emb, query_ids = load_separate(save_dir, "queries")

query1 = get_colbert(query_emb, 0) # [14 x 1024]
query1_smve = smve(query1, k, B, is_query=True)

smve_similarity = []

# docs smve for every doc with loop
for i in range(100):
    doc_vecs = get_colbert(doc_emb, i) #  447 x 1024
    doc_vec_smve = smve(doc_vecs, k, B, is_query=False)
    similarity = query1_smve @ doc_vec_smve.T
    smve_similarity.append([similarity, doc_ids[i]])


scores = sorted(smve_similarity, key=lambda x: x[0], reverse=True)
best_doc_id = scores[0][1]
best_doc_score = scores[0][0]

# top 5 docs
top_5_docs = scores[:5]
top_5_doc_ids = [doc[1] for doc in top_5_docs]


print("Best doc id:", best_doc_id)
print("Best doc score:", best_doc_score)
print("Top 5 doc ids:", top_5_doc_ids)  
print("Top 5 doc scores:", [doc[0] for doc in top_5_docs])


Best doc id: 72159
Best doc score: tensor(1.5776)
Top 5 doc ids: ['72159', '301866', '106301', '7912', '4983']
Top 5 doc scores: [tensor(1.5776), tensor(1.2317), tensor(1.1459), tensor(1.1357), tensor(1.1129)]


In [30]:
d = 1024
w = 65536
R = 8              # Number of repetitions (blocks)
k = 4              # k per block (total non-zeros selected per token = R * k = 32)

# Ensure w is evenly divisible by R
assert w % R == 0, "w must be divisible by R"
w_block = w // R   # 8192 columns per block

B = torch.randn(d, w)
col_norms = B.norm(dim=0, keepdim=True)
B = B / col_norms  # Column-normalized anchors


def smve_with_repetitions(
    token_embeddings: torch.Tensor,
    k_per_block: int,
    R: int,
    B: torch.Tensor,
    is_query: bool
    ) -> torch.Tensor:

    if not isinstance(token_embeddings, torch.Tensor):
        token_embeddings = torch.from_numpy(np.array(token_embeddings, dtype=np.float16))
    token_embeddings = token_embeddings.to(B.dtype)

    n_tokens = token_embeddings.shape[0]
    w_total = B.shape[1]
    w_block = w_total // R

    # Step 1: Project onto all w anchors
    # Projections shape: [n_tokens x w]
    projections = token_embeddings @ B 

    # Step 2: Block-wise Sparsification (Repetition Logic)
    # Reshape projections to isolate blocks: [n_tokens, R, w_block]
    proj_blocks = projections.view(n_tokens, R, w_block)
    

    # Pick top-k PER BLOCK along the last dimension (-1)
    # Values & indices shapes: [n_tokens, R, k_per_block]
    block_values, block_indices = torch.topk(proj_blocks, k_per_block, dim=-1)

    # Scatter selected values back into a block-structured sparse tensor
    sparse_blocks = torch.zeros_like(proj_blocks)
    sparse_blocks.scatter_(dim=-1, index=block_indices, src=block_values)

    # Flatten back to global anchor space: [n_tokens x w]
    sparse = sparse_blocks.view(n_tokens, w_total)

    # Step 3: Pooling across token sequence
    pooled = sparse.sum(dim=0)

    if is_query:
        return pooled

    # Document-side averaging normalization per active anchor
    mask_blocks = torch.zeros_like(proj_blocks)
    mask_blocks.scatter_(dim=-1, index=block_indices, src=torch.ones_like(block_values))
    mask = mask_blocks.view(n_tokens, w_total)
    
    counts = mask.sum(dim=0).clamp(min=1)
    doc_vec = pooled / counts

    return doc_vec


# --- Test Script ---
save_dir = ARTIFACTS_DIR / "embeddings" / "scifact_bgem3"
doc_emb, doc_ids = load_separate(save_dir, "docs", mmap_colbert=True)
query_emb, query_ids = load_separate(save_dir, "queries")

query1 = get_colbert(query_emb, 0)  # [14 x 1024]
query1_smve = smve_with_repetitions(query1, k_per_block=k, R=R, B=B, is_query=True)

smve_similarity = []

# Documents SMVE scoring
for i in range(100):
    doc_vecs = get_colbert(doc_emb, i)  # [n_doc_tokens x 1024]
    doc_vec_smve = smve_with_repetitions(doc_vecs, k_per_block=k, R=R, B=B, is_query=False)
    
    similarity = query1_smve @ doc_vec_smve.T
    smve_similarity.append([similarity.item(), doc_ids[i]])

scores = sorted(smve_similarity, key=lambda x: x[0], reverse=True)
best_doc_id = scores[0][1]
best_doc_score = scores[0][0]

top_5_docs = scores[:5]

print(f"Repetitions (R={R}), k per block={k} (Total active anchors = {R * k}):")
print("Best doc id:", best_doc_id)
print("Best doc score:", best_doc_score)
print("Top 5 doc ids:", [doc[1] for doc in top_5_docs])
print("Top 5 doc scores:", [doc[0] for doc in top_5_docs])

Repetitions (R=8), k per block=4 (Total active anchors = 32):
Best doc id: 301838
Best doc score: 1.2762848138809204
Top 5 doc ids: ['301838', '4983', '301866', '5836', '79447']
Top 5 doc scores: [1.2762848138809204, 1.1904017925262451, 1.0866076946258545, 0.8972419500350952, 0.8773410320281982]
